# Step 4.g — SVM climate suitability, ERA5 and the four AMOC scenarios (EC-Earth3 grid, global)

For each crop, the SVM of 3.g (trained on ERA5 1981–2014 over the MIRCA2000 cropland, with the BIO variant
`CHOSEN_VARIANT` and the radiation `CHOSEN_RAD` of the tests) is applied to **every land cell**:
- **ERA5** (1981–2014): the BIO variables of 1.1.d, the reference climate;
- the **four AMOC scenarios** (2025 / 2050 stabilisation, original and hosing): the delta BIO variables of 1.3.a
  (ERA5 + EC-Earth3 change).

**Suitability probability, at equal class priors.** The SVM is trained with balanced class weights (3.g), but the
Platt scaling of libsvm (`predict_proba`) fits its sigmoid on the unweighted training cells, so the probabilities
reproduce the share of presences of each crop in its training table (maize: presences the majority, mean probability
0.78 on the cropland; rice: 0.32). The climate probability is later multiplied by the soil suitability (step 5), so
its scale must mean the same for every crop. With `BALANCED = True` the probability is brought to equal class priors
(prior correction of the odds, Saerens et al. 2002, *Neural Comput.* 14, 21–41; Elkan 2001, *IJCAI*):
logit p' = logit p − log(n_presences / n_absences) (`cu.balanced_proba`). It is a monotonic transform: the ranking of
the cells and the AUC do not change. The Platt probability stays in `suit_proba_platt`.

**Cross-fitting of the training cells** (`CROSSFIT = True`, since 6 November 2026, option A of test 8.a). The final
model has seen the training cells: on them it reproduces their label (in sample 0.3 % of the maize absences called
suitable, about 6 % out of sample), and only on ERA5, the climate it was trained on, so that the comparison of ERA5
with the runs overstated the losses of today's cropland by 1–3 points. Each training cell is therefore predicted, in
every climate, by the model of the fold of 3.g (5 folds, seed 42, cost and gamma of the final model) that did not see
it (`cu.crossfit_models`, `cu.crossfit_proba`; fold models saved next to the final models); the other cells keep the
final model. Same rule otherwise; test 8.a: the signs of the hosing effects are unchanged.

**Binary climate suitability** (`suit_bin`, `cu.climate_suitable`) = probability ≥ `cu.CLIMATE_P` (0.5, the decision
threshold of the classifier at equal priors; out of sample the TSS is maximal at 0.42–0.57 for every crop) AND
growing-period temperature ≥ the absolute minimum temperature TMIN of FAO ECOCROP (`cu.ECOCROP_TMIN`: maize 10,
potato 7, soybean 10, rice 10, wheat 5 °C; season mean, phase 2 for winter wheat and wheat, as 3.h), in every
scenario with its own temperatures: the physiological floor of a correlative model (Kearney & Porter 2009). Since
6 November 2026 the temperature tested is that of the lower part of the cell (`cu.tmin_warming`): cell mean + 6.5 °C
km⁻¹ × (mean − 10th percentile of the elevation, notebook 0.5), i.e. at least about 10 % of the cell warm enough. In
mountain cells the crop grows in the valleys (relief of the cultivated cells below TMIN: 390–640 m, against 60–90 m
for the others); on flat land the correction is 0, so the cold tail of the taiga does not come back. The SVM
is trained on the cropland only, so it can call a cold land outside it suitable (maize: 70 Mha north of 50° N below
TMIN on ERA5); the filter removes it at a cost of 0–0.6 point of the MIRCA2000 area and changes no hosing effect by
more than 0.3 point (test of 5 November 2026). The soil is added in steps 5–6 (`cu.suitable`). Areas in Mha of land
(cell area × land fraction of 0.1), and the MIRCA2000 harvested areas (crop, all crops) in the suitable cells.

**Climates outside the training (MESS).** The SVM is trained on the MIRCA2000 cropland only: it has never seen the
climates beyond the edge of today's agriculture (Antarctica, Greenland, central Sahara), and far from the training
cells an RBF SVM gives back its intercept (Antarctica "suitable" for maize). The multivariate environmental
similarity surface (MESS, Elith et al. 2010, *Methods Ecol. Evol.* 1, 330–342) measures, for each cell, how far each
BIO variable is from the values of the training cells of that crop (ERA5 1981–2014 cropland); MESS < 0 = at least one
variable outside the training range ("novel climate"). The MESS looks at one variable at a time: a cell whose
variables are each within their range but whose combination exists in no training cell is not flagged (winter wheat
on the coasts of Greenland in test 4). **NT2** of ExDet (Mesgaran et al. 2014, *Divers. Distrib.* 20, 1147–1159)
flags those: the Mahalanobis distance to the training climates, divided by the largest one among the training cells;
NT2 > 1 = novel combination. Neither sees a cell that lies within the cloud but far from every training cell: the
Arctic with the GGCMI calendar (potato, gamma 4: 99.7 % of the cold suitable cells north of 50 N at the intercept
probability 0.507, MESS > 0, NT2 < 1). There the RBF kernel reaches no support vector: the decision function
b + Σ a_i y_i K(x, sv_i) falls back to the intercept b. The **kernel support** Kmax = exp(−γ d²), d = distance to the
nearest support vector in the standardised predictor space of the SVM, γ = gamma of the model (`cu.kernel_support`),
flags those: Kmax < `KERNEL_EPS` (0.01: the nearest support vector weighs less than 1 % of its full kernel value).
Novel = MESS < 0 or NT2 > 1 or Kmax < 0.01. The same reference (ERA5) for the four scenarios, so a scenario climate
outside today's cultivated climates is flagged too.

The **area of applicability** (AOA, Meyer & Pebesma 2021, *Methods Ecol. Evol.* 12, 1620–1633: dissimilarity index
DI = distance to the nearest training cell / mean distance between training cells, threshold = upper whisker of the
DI of the training cells to the other folds of 3.g, `cu.aoa`) was tried as the third criterion and is kept as
information only (`di`, area outside the AOA in the tables). It does not separate the far field from climates the
SVM still predicts: in the Arctic potato cells DI / threshold has a median of 2.4 with Kmax ~ 5e-5, while in the 2050
runs it flags 16–31 % of today's maize, soybean and potato area and 58–64 % of rice, where Kmax has a median of 0.3–0.6
and the probabilities are far from the intercept value. No multiple k of the threshold removes the first without the
second (k = 1.5: 99 % of the potato tail, but 26 % of the 2050 rice area). With Kmax < 0.01: 99.4 % of the potato
tail removed, at most 2.5 % of the 2050 crop area (potato, 0.4 % or less for the other crops).

Novel cells are **outside the training domain**: the model cannot assess them. They get `suit_proba` 0 (so that a
product never counts them as suitable) and their own class in `suit_class` (0 not suitable, 1 suitable, 2 novel,
3 too cold: probability ≥ 0.5 but below the ECOCROP TMIN), and
the areas report them apart: a cell that becomes novel in a scenario is not counted as a loss of suitable land
(Elith et al. 2010 recommend flagging extrapolation rather than predicting through it). The hosing effect is also
given on the cells assessable in both scenarios of the pair.

Outputs in `<out_root>/4_svm_predict/<route>/<variant>_<radiation>/`: `suitability_<crop>.nc` (`suit_proba`,
`suit_bin`, `suit_class`, `novel`, `below_tmin`, `suit_proba_svm`, `suit_proba_platt`, `mess`, `mess_var`, `nt2`, `kmax`, `di`, dimension
`scenario`), `suitable_area_<...>.csv`, `amoc_effect_<...>.csv`, `novel_climate_<...>.csv` and maps.

In [ ]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import numpy as np, pandas as pd, xarray as xr, joblib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from pathlib import Path
from sklearn.metrics import roc_auc_score
import warnings; warnings.filterwarnings('ignore')

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())  # racine du projet (local ou BDAP)
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu   # helpers + config reader (project root)
CFG = cu.load_config()   # config/config_<profile>.json, written by 00_config_and_inventory.ipynb

# ---- parameters: the choices of the tests of 3.g
CHOSEN_VARIANT = "A_60d"           # test 1
CHOSEN_RAD = os.environ.get("CHOSEN_RAD", "no_rad")   # test 2; no_rad until the rsds of the hosing runs arrives (as 1.3.a);
                                                     # set for one run with the environment (CHOSEN_RAD=rad_era5)
CELL_AREA_THR = 0.0005            # test 4; None = route no_threshold (tests 1-3); 0.0005 = route cellareafrac, presence and
                                  # background = area > 0.05 % of the cell land area (test 4); same value in 1.1.d, 3.g, 1.3.a, 4.g
CROPS = ["Maize_rf", "Potato_rf", "Soybean_rf", "Rice_rf", "Wheat_rf", "Wheat-winter_rf", "Wheat-spring_rf"]
ERA5_NAME, ERA5_PERIOD = "ERA5", ("1981", "2014")
RUNS = ["gcm_2025_nohosing", "gcm_2025_hosing", "gcm_2050_nohosing", "gcm_2050_hosing"]
PAIRS = [("gcm_2025_hosing", "gcm_2025_nohosing"), ("gcm_2050_hosing", "gcm_2050_nohosing")]   # hosing - original
ERA5_ONLY = os.environ.get("ERA5_ONLY", "0") == "1"   # ERA5 alone (no delta BIO needed: rad_era5 before the hosing rsds)
if ERA5_ONLY:
    RUNS, PAIRS = [], []
BALANCED = True                    # Platt probability brought to equal class priors (cu.balanced_proba); False = Platt
THRESHOLD = cu.CLIMATE_P           # binary climate suitability: probability >= 0.5 and growing season >= ECOCROP TMIN
DT = cu.tmin_warming(CFG)          # degC, lower part of the cell for the TMIN test (0 if cu.TMIN_LAPSE is False)
KERNEL_EPS = 0.01                  # novel where Kmax < KERNEL_EPS: no support vector within the reach of the RBF kernel
CROSSFIT = True                    # training cells predicted by the fold model that did not see them (cu.crossfit_models)
N_JOBS = min(5, os.cpu_count() or 1)   # fold models fitted in parallel (only when they are not saved yet)
assert CHOSEN_VARIANT in ("A_60d", "B_season", "C_crop") and CHOSEN_RAD.removesuffix("_relief") in ("rad_era5", "no_rad", "rad_fao56"), \
    "set CHOSEN_VARIANT and CHOSEN_RAD to the choices of the tests of 3.g"

SET = f"{CHOSEN_VARIANT}_{CHOSEN_RAD}"
TAG = f"{ERA5_NAME}_{ERA5_PERIOD[0]}_{ERA5_PERIOD[1]}_ecearth3"
ROUTE = cu.route(CELL_AREA_THR)
ERA5_DIR = cu.out_dir(CFG, "1_biovariables", f"{ERA5_NAME}_{ROUTE}", CHOSEN_VARIANT)
DELTA = cu.out_dir(CFG, "1_biovariables", f"GCM_delta_{ROUTE}", SET.removesuffix("_relief"))   # RELIEF (test 6) is
                                                                                                # static: not in the BIO files
MODELS = cu.out_dir(CFG, "3_svm_train", f"{ERA5_NAME}_{ROUTE}", SET)
OUT = cu.out_dir(CFG, "4_svm_predict", ROUTE, SET)

GRID = cu.svm_grid(CFG)
LAND = cu.land_mask(CFG).values
CRO = cu.mirca_cropland(CFG, CELL_AREA_THR).values        # training background
LAND_MHA = np.nan_to_num(cu.land_area_ha(CFG)) * LAND / 1e6   # land area of the cells (Mha)
with xr.open_dataset(cu.out_dir(CFG, cu.MASKS) / "mirca2000_any_crop_ecearth3.nc") as ds:
    CROPLAND_HA = np.nan_to_num(ds["area_ha"].values)    # MIRCA2000 harvested area of all crops (ha)
SCEN = {"ERA5": None, **{r: r for r in RUNS}}
print(f"set {SET} | models {MODELS} | balanced probability {BALANCED} | scenarios ERA5 + {RUNS}\noutputs {OUT}")

## 1. Prediction

In [ ]:
def bio_file(scenario, crop):
    if scenario == "ERA5":
        return ERA5_DIR / f"BIOvariables_{TAG}_{crop}.nc"
    return DELTA / f"BIOvariables_delta_{scenario}_{crop}.nc"


def predict(bundle, path, ref, di_of, k_of):
    # Platt probability, MESS (and its most dissimilar variable), NT2, kernel support, DI of the AOA of every land cell
    with xr.open_dataset(path) as ds:
        X = cu.bio_array(ds, bundle["bio_cols"], CFG).reshape(-1, len(bundle["bio_cols"]))
    ok = np.isfinite(X).all(axis=1) & LAND.ravel()
    p, m, v, n, k, d = (np.full(X.shape[0], np.nan), np.full(X.shape[0], np.nan), np.full(X.shape[0], -1),
                        np.full(X.shape[0], np.nan), np.full(X.shape[0], np.nan), np.full(X.shape[0], np.nan))
    Xs = bundle["scaler"].transform(X[ok])
    p[ok] = bundle["model"].predict_proba(Xs)[:, 1]
    m[ok], v[ok] = cu.mess(X[ok], ref)
    n[ok] = cu.nt2(X[ok], ref)
    k[ok] = k_of(Xs)
    d[ok] = di_of(Xs)
    return tuple(a.reshape(LAND.shape) for a in (p, m, v, n, k, d))


mha = lambda mask: float(np.nansum(LAND_MHA * mask))
rows, novel_rows, SUIT, NOVEL = [], [], {}, {}
for crop in CROPS:
    bundle = joblib.load(MODELS / f"svm_model_{TAG}_{crop}.joblib")
    n1, n0 = bundle["n_class1"], bundle["n_class0"]
    tr = pd.read_csv(ERA5_DIR / f"training_dataset_{TAG}_{crop}.csv")
    ii = np.searchsorted(GRID["lat"], tr.lat.values - 1e-6); jj = np.searchsorted(GRID["lon"], tr.lon.values - 1e-6)
    with xr.open_dataset(bio_file("ERA5", crop)) as ds:  # training cells: the climates the SVM has seen (from the
        REF = cu.bio_array(ds, bundle["bio_cols"], CFG)[ii, jj]                       # netCDF, not the rounded csv)
    fin = np.isfinite(REF).all(axis=1)
    REF = REF[fin]
    di_of, DI_THR = cu.aoa(bundle["scaler"].transform(REF), tr.label.values[fin])   # folds of 3.g (5, seed 42), information
    R3 = {s: predict(bundle, bio_file(s, crop), REF, di_of, cu.kernel_support(bundle["model"])) for s in SCEN}
    stack = lambda k: xr.DataArray(np.stack([R3[s][k] for s in SCEN]), dims=("scenario", "lat", "lon"),
                                   coords={"scenario": list(SCEN), "lat": GRID["lat"], "lon": GRID["lon"]})
    p_platt, mss, mvar, nt, kmax, di = stack(0), stack(1), stack(2), stack(3), stack(4), stack(5)
    if CROSSFIT:                                          # no memorisation of the labels of the training cells
        Fd = cu.crossfit_models(MODELS / f"folds_{TAG}_{crop}.joblib", tr[bundle["bio_cols"]].values.astype(float),
                                tr["label"].values.astype(int), bundle, n_jobs=N_JOBS)
        pv = p_platt.values.copy()
        for k, s in enumerate(SCEN):
            with xr.open_dataset(bio_file(s, crop)) as ds:
                g = cu.crossfit_proba(Fd, cu.bio_array(ds, bundle["bio_cols"], CFG)[ii, jj])
            fin = np.isfinite(g)
            pv[k, ii[fin], jj[fin]] = g[fin]
        p_platt = p_platt.copy(data=pv)
    p_svm = cu.balanced_proba(p_platt, n1, n0) if BALANCED else p_platt
    novel = ((mss < 0) | (nt > 1) | (kmax < KERNEL_EPS)) & p_svm.notnull()
    proba = xr.where(novel, 0.0, p_svm)                   # novel climate -> 0 (NaN off land stays NaN)
    tmin, which = cu.ECOCROP_TMIN[crop]
    tg = []
    for s in SCEN:                                        # growing-period temperature of every scenario (ECOCROP)
        with xr.open_dataset(bio_file(s, crop)) as ds:
            tg.append(np.asarray(cu.growing_temperature(ds, which), float))
    tgrow = xr.DataArray(np.stack(tg), dims=("scenario", "lat", "lon"), coords=proba.coords)
    cold = tgrow.notnull() & (tgrow + DT < tmin)           # lower part of the cell (cu.tmin_warming)
    suit = xr.where(proba.notnull(), xr.DataArray(cu.climate_suitable(proba.values, tgrow.values, crop, DT), coords=proba.coords).astype("float32"), np.nan)
    cls = xr.where(proba.notnull(), xr.where(novel, 2.0, xr.where((proba >= THRESHOLD) & cold, 3.0, suit)), np.nan)
    xr.Dataset({"suit_proba": proba.astype("float32").assign_attrs(
                    long_name="climate suitability probability" + (", equal class priors" if BALANCED else ", Platt"),
                    note=f"0 where the climate is novel (MESS < 0 or NT2 > 1 or Kmax < {KERNEL_EPS}), see suit_class"),
                "suit_bin": suit.assign_attrs(long_name="binary climate suitability: probability >= threshold and growing season >= ECOCROP TMIN",
                                              threshold=THRESHOLD, ecocrop_tmin=tmin, ecocrop_temperature=which,
                                              tmin_lapse=int(cu.TMIN_LAPSE), lapse_rate=cu.LAPSE_RATE),
                "suit_class": cls.astype("float32").assign_attrs(long_name="0 not suitable, 1 suitable, 2 novel climate (outside the training domain), 3 too cold (probability >= threshold, below the ECOCROP TMIN)"),
                "below_tmin": cold.astype("int8").assign_attrs(long_name=f"growing-period temperature ({which}) of the lower part of the cell below the ECOCROP TMIN {tmin} degC"),
                "novel": novel.astype("int8").assign_attrs(long_name=f"novel climate: MESS < 0 or NT2 > 1 or Kmax < {KERNEL_EPS}"),
                "suit_proba_svm": p_svm.astype("float32").assign_attrs(long_name="SVM probability, no MESS",
                                                                       balanced=str(BALANCED)),
                "suit_proba_platt": p_platt.astype("float32").assign_attrs(long_name="Platt probability of libsvm (prior = share of presences of the training table)"),
                "mess": mss.astype("float32").assign_attrs(long_name="MESS (Elith et al. 2010), reference ERA5 training cells",
                                                           note="< 0: at least one BIO variable outside the training range"),
                "mess_var": mvar.astype("int16").assign_attrs(long_name="most dissimilar variable (index in bio_cols)",
                                                              bio_cols=", ".join(bundle["bio_cols"]), missing=-1),
                "nt2": nt.astype("float32").assign_attrs(long_name="NT2 of ExDet (Mesgaran et al. 2014), reference ERA5 training cells",
                                                         note="> 1: combination of variables outside the training cloud"),
                "kmax": kmax.astype("float32").assign_attrs(long_name="kernel support: largest RBF kernel value to the support vectors, exp(-gamma d^2)",
                                                            threshold=KERNEL_EPS, note="< threshold: no support vector within reach, the SVM gives back its intercept (novel)"),
                "di": di.astype("float32").assign_attrs(long_name="dissimilarity index of the area of applicability (Meyer & Pebesma 2021), reference ERA5 training cells",
                                                        threshold=DI_THR, note="> threshold: outside the area of applicability; information only, not a novelty criterion")},
               attrs={"crop": crop, "set": SET, "model": f"one-pass SVM, cost {bundle['best_cost']}, gamma {bundle['best_gamma']}",
                      "bio_cols": ", ".join(bundle["bio_cols"]), "presences": n1, "absences": n0,
                      "balanced": str(BALANCED), "kernel_eps": KERNEL_EPS, "aoa_threshold": DI_THR}).to_netcdf(OUT / f"suitability_{crop}.nc")
    # in-sample check on the training cells (not a validation: the cross-validation of 3.g is)
    auc_in = roc_auc_score(tr.label.values, proba.sel(scenario="ERA5").values[ii, jj])
    crop_ha = np.nan_to_num(cu.crop_mask(CFG, crop)["area_ha"].values)
    SUIT[crop] = {s: suit.sel(scenario=s).values == 1 for s in SCEN}
    NOVEL[crop] = {s: novel.sel(scenario=s).values for s in SCEN}
    era5_area = mha(SUIT[crop]["ERA5"])
    for s in SCEN:
        b, nv = SUIT[crop][s], NOVEL[crop][s]
        a = mha(b)
        rows.append({"crop": crop, "scenario": s, "presence share": n1 / (n1 + n0),
                     "suitable land (Mha)": a,
                     "change vs ERA5 (%)": 100 * (a - era5_area) / era5_area if s != "ERA5" else 0.0,
                     "suitable land in cropland cells (Mha)": mha(b & CRO),
                     "MIRCA cropland harvested area in suitable cells (Mha)": CROPLAND_HA[b].sum() / 1e6,
                     "MIRCA area of the crop in suitable cells (%)": 100 * crop_ha[b].sum() / crop_ha.sum(),
                     "novel climate (Mha)": mha(nv),
                     "novel in cropland cells (Mha)": mha(nv & CRO),
                     "MIRCA area of the crop in novel cells (%)": 100 * crop_ha[nv].sum() / crop_ha.sum(),
                     "outside the AOA, DI > threshold, information (Mha)": mha(di.sel(scenario=s).values > DI_THR),
                     "MIRCA area of the crop outside the AOA, information (%)": 100 * crop_ha[di.sel(scenario=s).values > DI_THR].sum() / crop_ha.sum(),
                     "SVM alone, no MESS (Mha)": mha(p_svm.sel(scenario=s).values >= THRESHOLD),
                     "removed by the ECOCROP TMIN (Mha)": mha((proba.sel(scenario=s).values >= THRESHOLD) & cold.sel(scenario=s).values),
                     "mean probability on cropland": float(np.nanmean(proba.sel(scenario=s).values[CRO])),
                     "mean Platt probability on cropland": float(np.nanmean(p_platt.sel(scenario=s).values[CRO])),
                     "in-sample AUC (ERA5)": auc_in if s == "ERA5" else np.nan})
        # which variable puts the cells outside the training range (MESS < 0), and novel combinations only (NT2)
        v, mneg, n2 = mvar.sel(scenario=s).values, mss.sel(scenario=s).values < 0, nt.sel(scenario=s).values > 1
        for name, sel in [("NT2 > 1 (combination)", nv & ~mneg & n2), (f"Kmax < {KERNEL_EPS} (no support vector within kernel reach)", nv & ~mneg & ~n2)]:
            if sel.any():
                novel_rows.append({"crop": crop, "scenario": s, "variable": name,
                                   "novel area (Mha)": mha(sel), "of which in cropland cells (Mha)": mha(sel & CRO)})
        for k, c in enumerate(bundle["bio_cols"]):
            sel = mneg & (v == k)
            if sel.any():
                novel_rows.append({"crop": crop, "scenario": s, "variable": c, "novel area (Mha)": mha(sel),
                                   "of which in cropland cells (Mha)": mha(sel & CRO)})
    print(f"{crop}: presences {n1 / (n1 + n0):.0%}, ERA5 {era5_area:.0f} Mha suitable, in-sample AUC {auc_in:.3f}, "
          f"mean probability on cropland {rows[-len(SCEN)]['mean probability on cropland']:.2f} "
          f"(Platt {rows[-len(SCEN)]['mean Platt probability on cropland']:.2f}), AOA threshold DI {DI_THR:.3f} (information)", flush=True)
area = pd.DataFrame(rows).set_index(["crop", "scenario"])
area.to_csv(OUT / f"suitable_area_{SET}.csv")
novel_tab = pd.DataFrame(novel_rows, columns=["crop", "scenario", "variable", "novel area (Mha)",
                                              "of which in cropland cells (Mha)"]).set_index(["crop", "scenario", "variable"])
novel_tab.to_csv(OUT / f"novel_climate_{SET}.csv")
with pd.option_context("display.max_rows", 200, "display.max_columns", 30, "display.width", 250):
    display(area.round(2))
    print("novel climate: MESS < 0 by most dissimilar variable, and NT2 > 1 only"); display(novel_tab.round(2))

## 2. Effect of the AMOC collapse: hosing − original, per stabilisation level

On all land (novel cells count as not suitable) and on the cells assessable in both scenarios of the pair (novel in
neither): the second is the effect the model can actually tell.

In [ ]:
if not PAIRS:
    print("ERA5 only: no AMOC pair")
else:
    rows = []
    for crop in CROPS:
        for hos, orig in PAIRS:
            a_h, a_o = mha(SUIT[crop][hos]), mha(SUIT[crop][orig])
            both = ~(NOVEL[crop][hos] | NOVEL[crop][orig])
            b_h, b_o = mha(SUIT[crop][hos] & both), mha(SUIT[crop][orig] & both)
            rows.append({"crop": crop, "pair": f"{hos} - {orig}", "area change (Mha)": a_h - a_o,
                         "area change (%)": 100 * (a_h - a_o) / a_o if a_o else np.nan,
                         "area change, assessable in both (Mha)": b_h - b_o,
                         "area change, assessable in both (%)": 100 * (b_h - b_o) / b_o if b_o else np.nan,
                         "novel in either (Mha)": mha(~both & LAND)})
    amoc = pd.DataFrame(rows).set_index(["crop", "pair"])
    amoc.to_csv(OUT / f"amoc_effect_{SET}.csv")
    with pd.option_context("display.width", 250):
        display(amoc.round(2))

    grey = ListedColormap(["#9e9e9e"])
    for crop in CROPS:
        ds = xr.open_dataset(OUT / f"suitability_{crop}.nc")
        nov = ds.novel == 1
        fig, axes = plt.subplots(2, 3, figsize=(20, 7))
        for ax, s in zip(axes[0], ["ERA5", PAIRS[0][1], PAIRS[1][1]]):
            ds.suit_proba.sel(scenario=s).where(~nov.sel(scenario=s)).plot(ax=ax, vmin=0, vmax=1, cmap="YlGn")
            nov.sel(scenario=s).where(nov.sel(scenario=s)).plot(ax=ax, cmap=grey, add_colorbar=False)
            ax.set_title(f"{crop}: {s} (grey: climate never seen in training)", fontsize=9)
        ds.mess.sel(scenario="ERA5").clip(-100, 100).plot(ax=axes[1, 0], vmin=-100, vmax=100, cmap="PuOr")
        axes[1, 0].set_title(f"{crop}: MESS ERA5 (< 0 = novel climate)", fontsize=9)
        for ax, (hos, orig) in zip(axes[1, 1:], PAIRS):
            either = nov.sel(scenario=hos) | nov.sel(scenario=orig)
            (ds.suit_proba.sel(scenario=hos) - ds.suit_proba.sel(scenario=orig)).where(~either).plot(ax=ax, vmin=-0.5, vmax=0.5, cmap="RdBu")
            either.where(either).plot(ax=ax, cmap=grey, add_colorbar=False)
            ax.set_title(f"{hos} - {orig} (grey: climate never seen in training in either)", fontsize=9)
        for ax in axes.ravel():
            ax.set_xlabel(""); ax.set_ylabel("")
        plt.tight_layout(); plt.savefig(OUT / f"suitability_maps_{SET}_{crop}.png", dpi=80); plt.show()